# Build a Transcript App

## Overview

You've learned how to work with text-based AI models, manage conversations, and use various APIs. But what about audio? What if you want to process spoken content, such as lectures, podcasts, interviews, etc., and transform them into structured, searchable text and comprehensive notes?

In this activity, you'll build a complete audio transcription and note-generation pipeline using OpenAI's Whisper API and GPT models. You'll explore:

1. **Why audio transcription matters** in the context of AI agents and applications
2. **How to split large audio files** into manageable segments
3. **How to use Whisper API** to convert speech to text
4. **How to process and combine** multiple transcripts
5. **How to generate structured notes** from raw transcripts using advanced prompt engineering

By the end, you'll have a working system that can take any lecture audio and produce professional-quality study notes automatically.

## The Bigger Picture: Voice Agents and Transcription

Before we dive into code, let's understand where audio transcription fits in the landscape of AI applications.

### Two Architectures for Voice Agents

When building AI systems that interact with voice, there are two main approaches:

1. **Speech-to-Speech (S2S)**: Multimodal models that directly process and output speech without relying solely on text transcripts
   - More conversational and natural
   - Captures tone, pacing, and nuance
   - Great for customer service, triage, and onboarding

2. **Speech-to-Text -> Text Processing -> Text-to-Speech (STT/TTS)**: Chain-based architecture using text as an intermediate representation
   - **This is what we're building in this notebook**
   - Produces a transcript as an artifact
   - Allows for analysis, modification, and guardrails
   - Easier to integrate with text LLM workflows and tool use
   - Enables straightforward safety checks and policy enforcement on text

The system you build today represents the "STT" part of a complete voice agent architecture. You'll take audio, convert it to text, and then use that text to create value out of it by processing it (in this case, comprehensive notes).

## Initial Setup

Let's import the libraries we'll need. We're using a few new libraries for this activity.

#### Run the Following Cell

In [2]:
# === Imports ===

import os                            # For interacting with the operating system (file paths, folders)
import glob                          # For finding files matching a specific pattern
import warnings                      # To manage warning messages (we'll suppress some syntax ones)
import markdown                      # To convert Markdown text to HTML
import ffmpeg                        # For audio manipulation (splitting MP3s)
from openai import OpenAI
from IPython.display import display, HTML, Markdown as IPMarkdown

Now let's initialize the OpenAI client and configure our environment:

#### Run the Following Cell

In [3]:
# === OpenAI Client Setup ===
client = OpenAI()

## Understanding the Challenge: Audio File Size Limits

The Whisper API has a file size limit of approximately **25MB** per request. A typical 1-hour lecture recorded at reasonable quality might be 50-100MB or more.

**The problem**: You can't send the entire lecture file to Whisper in one request.

**The solution**: Split the audio into smaller segments, transcribe each segment separately, then combine the results.

This is a common pattern in working with such APIs: when you hit a limit, you break the problem into smaller pieces and process them sequentially.

## Part 1: Splitting Audio Files

We'll use the `ffmpeg` library to split our audio file into segments. The library expects segment lengths in seconds (10 minutes = 600 seconds).

Let's define a function that:
1. Loads an MP3 file
2. Splits it into chunks of specified fixed-length segments
3. Saves each chunk as a separate MP3 file

#### Run the Following Cell

In [4]:
def split_audio_file(input_file: str, output_folder: str, segment_length_minutes: int = 10) -> bool:
    """
    Splits an audio file into segments of specified duration.
    
    Args:
        input_file: Path to the source MP3 file
        output_folder: Directory to save the segments
        segment_length_minutes: Length of each segment in minutes
    
    Returns:
        True if successful, False otherwise
    """
    print(f"\n--- Starting Audio Splitting ---")
    print(f"Input: {input_file}")
    print(f"Output folder: {output_folder}")
    print(f"Segment length: {segment_length_minutes} minutes")
    
    if not os.path.isfile(input_file):
        print(f"Error: Input file not found at '{input_file}'")
        return False
    
    try:
        print("Loading audio...")
    
        # Create output directory
        os.makedirs(output_folder, exist_ok=True)
    
        segment_length_seconds = segment_length_minutes * 60
        output_pattern = os.path.join(output_folder, "segment_%03d.mp3")
    
        print("Splitting audio with ffmpeg...")
    
        (
            ffmpeg
            .input(input_file)
            .output(
                output_pattern,
                f="segment",
                segment_time=segment_length_seconds,
                c="copy"
            )
            .run(quiet=True)
        )
    
        # Count created segments
        segment_count = len([
            f for f in os.listdir(output_folder)
            if f.startswith("segment_") and f.endswith(".mp3")
        ])
    
        print(f"\n✓ Created {segment_count} segments")
        return True

    except Exception as e:
        print(f"Error during splitting: {e}")
        return False

Now let's configure our file paths. We'll set up directories for segments, transcripts, and final notes:

#### Run the Following Cell

In [5]:
# Input audio file
lecture_audio_file = "data/audio_input/lecture-04.mp3"

# Intermediate storage
segments_folder = "data/segments"
transcripts_folder = "data/transcriptions"

# Output
combined_transcript_file = os.path.join(transcripts_folder, "combined_transcript.txt")
notes_output_folder = "data/lecture_notes"

# Parameters
segment_length_min = 10
ai_model = "gpt-4o"

Let's test the splitting function with our lecture audio:

#### Run the Following Cell

In [6]:
split_successful = split_audio_file(
    input_file=lecture_audio_file,
    output_folder=segments_folder,
    segment_length_minutes=segment_length_min
)

if split_successful:
    print("\n✓ Audio splitting completed successfully!")
else:
    print("\n✗ Audio splitting encountered errors.")


--- Starting Audio Splitting ---
Input: data/audio_input/lecture-04.mp3
Output folder: data/segments
Segment length: 10 minutes
Loading audio...
Splitting audio with ffmpeg...

✓ Created 7 segments

✓ Audio splitting completed successfully!


### What Just Happened?

The function loaded your audio file and split it into manageable chunks. Each chunk is exactly 10 minutes (or less for the final segment).

**Behind the scenes:**
1. We computed the segment length in seconds (10 minutes = 600 seconds).
2. `ffmpeg` split the audio into fixed-length segments using its built-in segmenting feature.
3. Each segment was exported as a new MP3 file with a sequential name (e.g., segment_001.mp3, segment_002.mp3)


💡 **Key insight**: Breaking large files into smaller pieces is a fundamental pattern in data processing. When you hit API limits, storage limits, or memory constraints, splitting the problem is a potential solution. In fact, if the files don't need to be processed sequentially, they can be processed quite efficiently using parallel processing.

## Part 2: Transcribing Audio With Whisper

## Understanding Whisper: OpenAI's Speech Recognition Model

Now that we have manageable audio segments, let's transcribe them using OpenAI's **Whisper** model.

### What is Whisper?

Whisper is OpenAI's automatic speech recognition (ASR) system, trained on 680,000 hours of multilingual and multitask supervised data collected from the web. It's remarkably robust and versatile:

**Key capabilities:**
- **Multilingual**: Supports various languages, including English, Spanish, French, German, Chinese, Japanese, and many more
- **Accent-tolerant**: Handles various accents and speaking styles
- **Noise-robust**: Works well even with background noise, music, or poor audio quality
- **Format-flexible**: Accepts MP3, MP4, MPEG, MPGA, M4A, WAV, and WEBM formats
- **Technical vocabulary**: Handles technical jargon, proper nouns, and domain-specific terms surprisingly well

**How it works:**
1. You send an audio file to the API
2. Whisper analyzes the audio waveform
3. It identifies speech patterns, phonemes, and words
4. It returns the transcribed text as a string

The process is straightforward: you open an audio file, send it to Whisper, and receive back the text of what was said.

## The Transcription Function

Let's create a function that transcribes a single audio segment. The function will:
1. Open an audio file in binary mode (`"rb"`)
2. Send it to the Whisper API using `client.audio.transcriptions.create()`
3. Extract the `.text` attribute from the response
4. Save the text to a file

Here's the complete function:

#### Run the Following Cell

In [7]:
def transcribe_audio_segment(audio_path: str, output_path: str) -> bool:
    """
    Transcribes a single audio segment using Whisper API.
    
    Args:
        audio_path: Path to the audio file
        output_path: Path to save the transcript
    
    Returns:
        True if successful, False otherwise
    """
    print(f"Transcribing: {os.path.basename(audio_path)}")
    
    try:
        # Open the audio file in binary mode
        with open(audio_path, "rb") as audio_file:
            # Send to Whisper API for transcription
            transcription = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file
            )
        
        # Extract the transcribed text
        text = transcription.text
        
        # Save transcript to file
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(text)
        
        print(f"  ✓ Saved to {os.path.basename(output_path)}")
        return True
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        return False

print("Transcription function defined successfully!")

Transcription function defined successfully!


### What Just Happened?

We created a function that handles the entire transcription workflow:

1. **Opens the file**: Using `open(audio_path, "rb")` opens the audio in binary read mode
2. **Calls Whisper**: `client.audio.transcriptions.create()` sends the audio to OpenAI's servers
3. **Specifies the model**: `model="whisper-1"` tells the API which Whisper model to use
4. **Extracts text**: The `.text` attribute contains the transcribed speech
5. **Saves to file**: Writes the transcript to a text file for later use

The Whisper API does all the heavy lifting of analyzing audio waves, identifying phonemes, recognizing words, and handling punctuation.

💡 **Key insight**: Speech recognition has become incredibly accessible. A few lines of code can now transcribe hours of audio with impressive accuracy, handling multiple languages and difficult audio conditions.

## Exercise 1: Transcribe All Segments

Now let's use our transcription function to process all the audio segments. Your task is simple: specify how many segments were successfully transcribed.

In the code below, print the number of successfully transcribed segments using the `success_count` variable.

Enter your answer between these comments in the ungraded code cell below:

```python
# YOUR CODE HERE
raise NotImplementedError("Your code is missing.")
# END OF YOUR CODE

#### Enter Your Solution Then Run the Following Cell

**Note: The cell may take a while to run**

In [8]:
print("\n--- Transcribing Audio Segments ---\n")

# Get all segment files
segment_files = sorted([f for f in os.listdir(segments_folder) if f.endswith(".mp3")])

if not segment_files:
    print("Error: No segment files found!")
else:
    print(f"Found {len(segment_files)} segments to transcribe\n")
    
    # Create transcripts folder
    os.makedirs(transcripts_folder, exist_ok=True)
    
    # Transcribe each segment
    success_count = 0
    for segment_file in segment_files:
        audio_path = os.path.join(segments_folder, segment_file)
        transcript_file = segment_file.replace(".mp3", ".txt")
        transcript_path = os.path.join(transcripts_folder, transcript_file)
        
        if transcribe_audio_segment(audio_path, transcript_path):
            success_count += 1
    
    # YOUR CODE HERE
    print(f"\n✓ Successfully transcribed {success_count} segments")
    # END OF YOUR CODE


--- Transcribing Audio Segments ---

Found 7 segments to transcribe

Transcribing: segment_000.mp3
  ✓ Saved to segment_000.txt
Transcribing: segment_001.mp3
  ✓ Saved to segment_001.txt
Transcribing: segment_002.mp3
  ✓ Saved to segment_002.txt
Transcribing: segment_003.mp3
  ✓ Saved to segment_003.txt
Transcribing: segment_004.mp3
  ✓ Saved to segment_004.txt
Transcribing: segment_005.mp3
  ✓ Saved to segment_005.txt
Transcribing: segment_006.mp3
  ✓ Saved to segment_006.txt

✓ Successfully transcribed 7 segments


<details>
<summary style="text-align: left; background-color: #e3fbe3; color: #2a8bc6; padding: 10px 10px; cursor: pointer;" role="alert"><strong>✅ Solution</strong></summary>

<p style="padding: 10px;">
Replace the print statement with:

<pre><code>print(f"\n✓ Successfully transcribed {success_count} segments")</code></pre>

This will display how many segments were successfully transcribed.
</p>

</details>

### What Just Happened?

You just converted multiple audio segments into searchable text! The code:

1. **Found all MP3 files** in the segments folder
2. **Looped through each file** one at a time  
3. **Called the transcription function** for each segment
4. **Tracked success** by counting how many transcriptions completed without errors
5. **Saved each transcript** as a `.txt` file

Each audio file was sent to Whisper, which analyzed the speech and returned text. The transcripts are now saved and ready to be combined.

This pattern of processing a collection of files one by one is fundamental in data processing. You'll see it everywhere: batch image processing, document analysis, data transformation pipelines. If your workflow allows for it, you can also try parallel processing while ensuring that you respect the rate limits set by your model developer and/or provider.

## Part 3: Combining Transcripts

We now have multiple text files, one for each audio segment. To generate comprehensive notes, we need to combine them into a single transcript.

This is straightforward: we just read each file and join them together. The key is to maintain the correct order (hence why we sorted the files earlier).

#### Run the Following Cell

In [9]:
def combine_transcripts(folder: str, output_file: str) -> str:
    """
    Combines multiple transcript files into one.
    
    Args:
        folder: Directory containing transcript .txt files
        output_file: Path to save the combined transcript
    
    Returns:
        The combined transcript text
    """
    print("\n--- Combining Transcripts ---\n")
    
    # Find all transcript files and sort them
    transcript_files = sorted(glob.glob(os.path.join(folder, "*.txt")))
    
    if not transcript_files:
        print("Error: No transcript files found!")
        return None
    
    print(f"Found {len(transcript_files)} transcripts to combine")
    
    # Read and combine
    all_content = []
    for file_path in transcript_files:
        with open(file_path, 'r', encoding="utf-8") as f:
            all_content.append(f.read())
    
    combined = "\n".join(all_content)
    
    # Save combined transcript
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(combined)
    
    print(f"✓ Combined transcript saved to {output_file}")
    print(f"  Total length: {len(combined):,} characters")
    
    return combined

Let's combine our transcripts:

#### Run the Following Cell

In [10]:
full_transcript = combine_transcripts(transcripts_folder, combined_transcript_file)

if full_transcript:
    print("\n--- Preview of Combined Transcript ---")
    print(full_transcript[:500] + "...")


--- Combining Transcripts ---

Found 8 transcripts to combine
✓ Combined transcript saved to data/transcriptions/combined_transcript.txt
  Total length: 183,427 characters

--- Preview of Combined Transcript ---
I should have a goal, but just showing that I guess the other question I wanted to say is how many of you missed week one? Okay, and the real question for this quiz is, reading the materials that are posted on Canvas, were you able to answer this quiz? Okay, so we're going to have these quizzes that are going to be practicing with flavor. And the concept of flavor is things that when you walk away from this class, you should know how to answer. So it will be really hard, like foundations, concep...


## Part 4: Generating Notes With AI

Now for the exciting part! We have a complete transcript. Let's use GPT to transform this raw transcript into structured, comprehensive lecture notes.

This is where **prompt engineering** becomes crucial. The quality of your notes depends heavily on how you instruct the AI.

### Experiment 1: Simple Prompt

Let's start with a simple prompt and see what we get:

#### Run the Following Cell

In [11]:
def generate_notes(transcript: str, system_prompt: str, user_prompt: str, model: str = "gpt-4o") -> str:
    """
    Generates notes from a transcript using AI.
    
    Args:
        transcript: The full transcript text
        system_prompt: System-level instructions
        user_prompt: Specific task instructions
        model: The AI model to use
    
    Returns:
        Generated notes in Markdown format
    """
    print(f"\n--- Generating Notes (using {model}) ---\n")
    
    if not transcript:
        print("Error: No transcript provided!")
        return None
    
    try:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{user_prompt}\n\nTranscript:\n{transcript}"}
        ]
        
        print("Sending request to OpenAI...")
        response = client.chat.completions.create(
            model=model,
            messages=messages
        )
        
        notes = response.choices[0].message.content
        print("✓ Notes generated successfully")
        return notes
        
    except Exception as e:
        print(f"Error generating notes: {e}")
        return None

Let's try a simple prompt:

#### Run the Following Cell

In [12]:
simple_system = "You are an expert academic assistant."
simple_user = "Please generate detailed lecture notes from the following transcript."

simple_notes = generate_notes(full_transcript, simple_system, simple_user, ai_model)


--- Generating Notes (using gpt-4o) ---

Sending request to OpenAI...
✓ Notes generated successfully


Let's see what we got:

#### Run the Following Cell

In [13]:
if simple_notes:
    print("--- Simple Prompt Results ---\n")
    display(IPMarkdown(simple_notes[:2000] + "\n\n...(truncated for preview)..."))

--- Simple Prompt Results ---



### Lecture Notes: Building AI Applications for Lecture Note Generation

#### Introduction
- **Objective**: Explore how to use AI for generating structured lecture notes from audio recordings.
- **Context**: Traditional slides might not capture all the nuances of a lecture.
- **Approach**: Develop a custom AI application to generate lecture notes, ensuring control over what information is included.

#### Key Points from the Lecture

1. **Course Outline and Quizzes**
   - The course will have quizzes designed to test foundational concepts.
   - Students need to read materials posted on Canvas prior to quizzes.

2. **Goal for Today's Lecture**
   - Utilize AI for creating detailed lecture notes.
   - Leverage AI for more than just transcription by generating rich, informative notes.

3. **Use of AI for Lecture Notes**
   - AI can supplement lecture material by capturing and structuring spoken content.
   - Tools like Zoom’s built-in AI were considered, but the course’s focus on AI applications led to the decision to build a custom solution.
   - Custom-built solutions offer greater control and customization potential.

4. **Recording Setup and Challenges**
   - Previous lectures were recorded but faced audio quality issues due to background noise and microphone placement.
   - Whisper by OpenAI chosen for audio-to-text conversion. It excels as it is trained on diverse voice-text datasets.

5. **Technical Approach**
   - **Whisper Model**: Specialized in converting audio to text. Not designed to understand text, purely transcription.
   - **Segmenting Audio**: Audio files need to be segmented to meet Whisper’s input size restrictions.
     - Experimented with audio slicing using a library called ‘AudioSegment’ from Hideout.
     - Managed quality by keeping segments around 10 minutes to fit within file size constraints.

6. **Coding and Implementation**
   - Code facilitated segmenting the audio and calling the Whisper API.
   - Scripts generated text files for each au

...(truncated for preview)...

### Experiment 2: Detailed Prompt

The simple prompt probably gave us decent notes, but we can do much better with specific instructions. Let's create a detailed prompt that specifies exactly what we want.

#### Run the Following Cell

In [14]:
detailed_system = """You are an expert academic assistant specializing in creating 
comprehensive, pedagogically sound lecture notes."""

detailed_user = """Please generate comprehensive lecture notes from the following transcript.

Incorporate these advanced learning techniques:

1. **Structure**: Use clear hierarchical organization (main topics → subtopics → key points)
2. **Summary**: Begin with learning objectives or key takeaways
3. **Formatting**: Use bullet points, numbered lists, and clear headings
4. **Emphasis**: Highlight key terms, definitions, and important concepts using **bold** or *italics*
5. **Examples**: Include relevant examples and real-world applications
6. **Engagement**: Add thought-provoking questions or discussion points
7. **Analogies**: Provide analogies or metaphors for difficult concepts
8. **Self-Check**: Insert brief "check your understanding" sections
9. **Memory Aids**: Include mnemonics where applicable
10. **Connections**: Add cross-references to related concepts
11. **Conclusion**: End with main takeaways and areas for further exploration

Feel free to expand on the content with relevant background information and fill in gaps 
to ensure completeness. The goal is comprehensive, engaging notes that facilitate deep 
understanding and retention.
"""

detailed_notes = generate_notes(full_transcript, detailed_system, detailed_user, ai_model)


--- Generating Notes (using gpt-4o) ---

Sending request to OpenAI...
✓ Notes generated successfully


In [15]:
if detailed_notes:
    print("--- Detailed Prompt Results ---\n")
    display(IPMarkdown(detailed_notes[:2000] + "\n\n...(truncated for preview)..."))

--- Detailed Prompt Results ---



### Lecture Notes: AI Applications for Generating Lecture Notes

---

### Key Learning Objectives
1. **Understand** the capabilities and limitations of the Whisper model for audio transcription.
2. **Develop** an AI application to generate structured lecture notes from audio recordings.
3. **Explore** the concepts of prompt engineering and its significance in AI applications.
4. **Discuss** real-world applications and challenges associated with AI tools.

---

### 1. Introduction to AI-generated Lecture Notes
- **Purpose**: The goal is to use AI to create lecture notes by transcribing audio into text, which can be structured into useful notes.
- **Context**: Traditional slides may not cover all information, and typical AI tools may not be optimized for lecture settings.

### 2. Overview of Whisper Model
- **Description**: Whisper is an AI model by OpenAI designed for converting audio to text.
- **Capabilities**: 
  - Handles large file sizes and can be segmented for processing.
  - Capable of adapting to various audio qualities, though origin audio quality can impact results.
- **Limitations**: No real-time conversion; intended for batch processing.

### 3. Development Process for Lecture Note Generation
#### A. Initial Setup
- **Audio Quality**: Challenges with poor audio quality (background noise, microphone issues).
- **Solution**: Audio segmentation using the `audio_segment` module to segment files into manageable sizes.

#### B. Preprocessing Steps
- **Audio Segmentation**: 
  - Use `AudioSegment` to split audio files into smaller segments before transcription.
  - Recognize that the segment size is critical to optimizing output quality.

#### C. Transcription
- **Whisper Model Utilization**:
  - Initialize OpenAI Whisper client; process each audio segment.
  - Ensure segments are in the correct order for accurate transcription.

### 4. Generating Lecture Notes from Transcripts
- **Text Summarization**: Use language models (e.g., GPT) to turn transcripts into s

...(truncated for preview)...

### What Just Happened?

We can see the how the prompt drastically shapes the results. 

**Simple prompt**: "Generate notes"
- Vague instruction
- AI makes its own decisions about structure and content
- Results are unpredictable

**Detailed prompt**: Specific requirements for structure, emphasis, examples, etc.
- Clear expectations
- AI knows exactly what you want
- Results are consistent and high-quality

The detailed prompt incorporates pedagogical best practices:
- **Hierarchical organization** helps learners build mental models
- **Examples** connect abstract concepts to concrete situations
- **Self-check questions** promote active learning
- **Memory aids** improve retention

💡 **Key insight**: Invest time in understanding how to prompt models effectively for your particular use cases. The effort pays off exponentially in output quality.

## Create Your Own Prompt

Now it's your turn to experiment with prompt engineering! You'll create a custom system prompt that shapes how the AI generates notes.

Think about a specific audience or purpose:
- Notes for a complete beginner (explain everything simply)
- Executive summary (focus on key decisions and implications)
- Study guide for an exam (include practice questions)
- Tutorial format (step-by-step with exercises)

In the code below, create a variable named `custom_system` and assign your system prompts to the variable. Try a single sentence or short paragraph.
Enclose the entire prompt in triple quotation marks (`"""`) so it is treated as a single string.

Enter your answer between these comments in the ungraded code cell below:

```python
# YOUR CODE HERE
raise NotImplementedError("Your code is missing.")
# END OF YOUR CODE

#### Enter Your Solution Then Run the Following Cell

In [16]:
# YOUR CODE HERE
custom_system = "You are an exam preparation tutor. Create study notes with clear definitions, important concepts highlighted, and practice questions after each section."
# END OF YOUR CODE

# Use the detailed user prompt from earlier
custom_user = detailed_user

# Generate notes with your custom prompt
custom_notes = generate_notes(full_transcript, custom_system, custom_user, ai_model)

if custom_notes:
    print("\n--- Your Custom Notes (Preview) ---\n")
    display(IPMarkdown(custom_notes[:2000] + "\n\n...(truncated for preview)..."))


--- Generating Notes (using gpt-4o) ---

Sending request to OpenAI...
✓ Notes generated successfully

--- Your Custom Notes (Preview) ---



**Lecture Notes: AI Application Class - Audio Transcription and Lecture Note Generation**

---

### **Learning Objectives:**
- Understand the process of using AI for generating lecture notes.
- Learn about handling audio files for transcription using Whisper by OpenAI.
- Explore the use of large language models (LLMs) to summarize and enrich lecture content.
- Discuss the potential and limitations of AI in transcription and summary generation.

---

### **1. Introduction to AI-Driven Lecture Note Generation**
- **Overview:**
  - The lecture focuses on automating lecture note generation using AI.
  - While traditional slides are shared, AI-generated notes can enhance learning.
- **Challenge:**
  - Manual note-taking is inefficient.
  - Existing tools like Zoom AI may not fully meet specific lecture needs.

### **2. Audio Transcription with Whisper**
- **Whisper Model:**
  - **Definition:** Whisper is a model provided by OpenAI for audio-to-text conversion.
  - **Purpose:** Transcribes lecture audio into raw text with high accuracy.
- **Handling Audio Files:**
  - **Issue:** Large audio files may exceed model input limits.
  - **Solution:** Use audio processing libraries (e.g., Audio Segment) to divide audio into smaller segments.
  - **Implementation:**
    - Break audio into 10-minute segments for optimal processing.
    - Utilize large Whisper model for better transcription accuracy.

### **3. Leveraging Large Language Models (LLMs)**
- **LLMs in Action:**
  - **Purpose:** transform raw transcripts into structured lecture notes.
  - **Example Models:** ChatGPT and Whisper.
- **Steps for Note Generation:**
  1. Divide audio into segments.
  2. Transcribe audio using Whisper.
  3. Process transcripts with LLMs to summarize and structure content.
- **Output Quality:**
  - Ensure content order by sequentially processing segments.
  - Generate coherent, structured notes that enhance the original lecture.

### **4. Advanced Techniques and Considerations**
- **Prompt Engi

...(truncated for preview)...

<details>
<summary style="text-align: left; background-color: #e3fbe3; color: #2a8bc6; padding: 10px 10px; cursor: pointer;" role="alert"><strong>✅ Example Solution</strong></summary>

<p style="padding: 10px;">
Here's one example for beginner-friendly notes:

<pre><code>custom_system = "You are a patient teacher who excels at explaining complex topics to complete beginners. Use simple language, everyday analogies, and define all technical terms."</code></pre>

Another example for a study guide:

<pre><code>custom_system = "You are an exam preparation tutor. Create study notes with clear definitions, important concepts highlighted, and practice questions after each section."</code></pre>
</p>

</details>

Experiment with different approaches to see how the same transcript can be transformed for different audiences.

## Saving and Displaying Notes

Let's create a function to save our notes in both Markdown and HTML formats, and display them nicely:

#### Run the Following Cell

In [17]:
def save_and_display_notes(notes: str, output_dir: str, filename: str = "lecture_notes"):
    """
    Saves notes as Markdown and HTML, and displays them.
    
    Args:
        notes: The notes content in Markdown format
        output_dir: Directory to save the files
        filename: Base filename (without extension)
    """
    if not notes:
        print("No notes to save.")
        return
    
    print(f"\n--- Saving Notes ---\n")
    os.makedirs(output_dir, exist_ok=True)
    
    # Save as Markdown
    md_path = os.path.join(output_dir, f"{filename}.md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(notes)
    print(f"✓ Markdown saved: {md_path}")
    
    # Convert to HTML and save
    html_content = markdown.markdown(notes)
    html_full = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>{filename}</title>
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; 
                line-height: 1.6; max-width: 800px; margin: 40px auto; padding: 20px; }}
        h1, h2, h3 {{ color: #333; margin-top: 1.5em; }}
        code {{ background-color: #f4f4f4; padding: 2px 6px; border-radius: 3px; }}
        pre {{ background-color: #f4f4f4; padding: 15px; border-radius: 5px; overflow-x: auto; }}
        blockquote {{ border-left: 4px solid #ddd; margin: 0; padding-left: 20px; color: #666; }}
    </style>
</head>
<body>
{html_content}
</body>
</html>"""
    
    html_path = os.path.join(output_dir, f"{filename}.html")
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html_full)
    print(f"✓ HTML saved: {html_path}")
    
    # Display in notebook
    print("\n--- Preview of Generated Notes ---\n")
    display(HTML(html_content))

Let's save and display our detailed notes:

#### Run the Following Cell

In [18]:
save_and_display_notes(detailed_notes, notes_output_folder, "Comprehensive_Lecture_Notes")


--- Saving Notes ---

✓ Markdown saved: data/lecture_notes/Comprehensive_Lecture_Notes.md
✓ HTML saved: data/lecture_notes/Comprehensive_Lecture_Notes.html

--- Preview of Generated Notes ---



## Putting It All Together: Complete Workflow

You've now learned all the pieces. Here's the complete workflow in one place, showing how all the parts connect:

```python
# 1. Split audio into manageable segments
split_audio_file(lecture_audio_file, segments_folder, segment_length_min)

# 2. Transcribe each segment with Whisper
for each segment:
    transcribe_audio_segment(segment, output)

# 3. Combine all transcripts into one
full_transcript = combine_transcripts(transcripts_folder, combined_file)

# 4. Generate comprehensive notes with AI
notes = generate_notes(full_transcript, system_prompt, user_prompt)

# 5. Save and display the results
save_and_display_notes(notes, output_folder)
```

This pipeline transforms **Raw audio** (50+ MB, unsearchable, unstructured) into **structured notes** (organized, searchable, study-ready).

## Reflection

By completing this activity, you've learned:

* **How to split large audio files** into manageable segments using pydub
* **How to use Whisper API** for accurate speech-to-text transcription
* **How to process and combine** multiple text files programmatically
* **How prompt engineering shapes output quality** during the text processing step
* **How to convert Markdown to HTML** for professional presentation

You've built a complete pipeline that transforms raw audio into searchable, structured notes.

**Think about:**

* **Where else could this be useful?** Could you apply this to podcasts, meetings, interviews, or customer calls? What would need to change for each use case?

* **What are the limitations?** How would you handle poor audio quality, multiple speakers, or very long recordings? What happens when Whisper misunderstands technical terms?

* **What could you build on top of this?** Could you add speaker identification? Generate flashcards automatically? Create searchable timestamps? Extract action items from meetings?

The system you built today is a foundation that can be extended in many directions. The key is understanding how the components work together.